# HyperRAG Tutorial: Step-by-Step Pipeline Walkthrough

**HyperRAG** is a Link-Graph Augmented Retrieval-Augmented Generation (RAG) system for multi-hop web question answering. Unlike standard RAG, which retrieves a fixed set of documents and stops there, HyperRAG exploits the native HTML hyperlinks between Wikipedia pages to *expand* retrieval into adjacent pages — following the same kind of reasoning path a human researcher would take when clicking links to find related information.

This tutorial walks through the complete HyperRAG pipeline in four stages:

1. **Stage 1 — Corpus Loading**: Load multi-hop questions from HotpotQA and build a corpus of Wikipedia pages.
2. **Stage 2 — Graph Construction**: Build a directed hyperlink graph (NetworkX DiGraph) from the `<a href>` links in the Wikipedia HTML pages.
3. **Stage 3 — Embeddings & FAISS Index**: Encode every corpus page as a dense vector using `sentence-transformers/all-MiniLM-L6-v2` and store them in a FAISS flat index for fast similarity search.
4. **Stage 4 — Retrieval Systems + Evaluation**: Compare three systems — B1 (Naive RAG), B2 (HtmlRAG-style), and HyperRAG — on real HotpotQA questions using Exact Match (EM) and F1 metrics.

**Who is this for?** Graders wanting to understand how each component works, and developers wanting to replicate or extend the system.

**What makes HyperRAG different?** Standard RAG and HtmlRAG retrieve pages independently. HyperRAG additionally follows the hyperlinks FROM those retrieved pages one hop outward — discovering pages the query didn't directly match but that a human reader following links would naturally reach. This 1-hop graph expansion is the core innovation, and it costs almost nothing: the hyperlink graph already *exists* inside the HTML `<a href>` tags. No LLM-extracted knowledge graph construction is needed.

## Setup

**Google Colab users:** Run the pip install cell below first, then run the imports cell.

**Local users:** Skip the pip install cell — all dependencies are already installed via `requirements.txt`.

In [ ]:
# Uncomment in Google Colab:
# !pip install -q transformers sentence-transformers faiss-cpu networkx datasets beautifulsoup4 requests numpy torch tqdm matplotlib

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import from src/
# Works whether running from notebooks/ dir (local) or the project root (Colab)
PROJECT_ROOT = Path('..') if Path('../src').exists() else Path('.')
sys.path.insert(0, str(PROJECT_ROOT))

# Stage 1 imports: HotpotQA loading and Wikipedia corpus management
from src.corpus import load_hotpotqa, load_corpus, build_corpus

# Stage 2 imports: NetworkX DiGraph construction and persistence
from src.graph import build_graph, build_and_save_graph, load_graph, print_graph_stats

# Stage 3 imports: sentence-transformers encoding and FAISS index
from src.embeddings import build_faiss_index, load_index, search

# Stage 4 imports: retrieval systems (B1, B2, HyperRAG) and evaluation metrics
from src.retrieval import naive_rag, htmlrag_style, hyperrag, compute_em, compute_f1

# Visualization and numeric libraries
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

print("All imports successful!")

## Stage 1: Corpus Loading

**What is HotpotQA?** HotpotQA (Yang et al., 2018) is a multi-hop question answering benchmark. Unlike simple QA datasets where a single paragraph contains the answer, HotpotQA questions require reasoning across *two or more* Wikipedia articles. For example, answering "What film starred the actor who played Moriarty in BBC Sherlock and was featured at Cannes 2003?" requires finding who played Moriarty, then finding what films that actor appeared in, then checking the Cannes film list — three hops of reasoning.

**Why do we need a corpus?** A RAG system doesn't answer questions from scratch; it retrieves relevant documents from a pre-built *document store*, then constructs an answer from the retrieved text. Our corpus is the document store: a collection of ~450 Wikipedia pages referenced by the first 500 HotpotQA questions. Without this corpus, there is nothing to retrieve from.

**What does each corpus page contain?** Each page is a Python dict with four fields:
- `title`: The Wikipedia article title (e.g. `"Albert Einstein"`).
- `text`: The plain text of the page (stripped of HTML tags). Used by B1 Naive RAG.
- `html`: The raw HTML of the page. Used by B2 HtmlRAG-style to preserve structure (tables, headings).
- `links`: A list of page titles this page links TO via `<a href>` tags. This is the raw material for the hyperlink graph in Stage 2.

**The caching pattern:** Fetching 450 Wikipedia pages from the Wikipedia API takes 20–40 minutes. To avoid re-fetching every session, `src/corpus.py` saves the result to `data/corpus.json` and checks for it on startup. If it exists, the corpus loads in under a second.

### Step 1.1: Load HotpotQA Questions

We load a small subset of HotpotQA training questions. Each item has a `question`, a gold `answer`, and `supporting_facts` — a list of (Wikipedia title, sentence index) pairs pointing to the exact Wikipedia pages and sentences the question was constructed from. The supporting facts tell us which pages are *relevant* to each question, which is how we compute EM/F1 at evaluation time.

In [ ]:
# Load 3 HotpotQA questions for our tutorial walkthrough
# We use n_samples=3 to keep execution fast — the full eval uses 50 questions
qa_items = load_hotpotqa(split="train", n_samples=3)

# Show the structure of each QA item
print(f"Loaded {len(qa_items)} questions\n")
for i, qa in enumerate(qa_items):
    print(f"Q{i+1}: {qa['question']}")
    print(f"    Answer: {qa['answer']}")
    # supporting_facts['title'] is a list of Wikipedia titles whose sentences support the answer
    # These are the pages HyperRAG must retrieve to score EM=1 on this question
    sf_titles = list(set(sf for sf in qa['supporting_facts']['title']))
    print(f"    Supporting pages: {sf_titles}\n")

### Step 1.2: Load the Corpus

The corpus was pre-built by `src/corpus.py` during Phase 2 of development. The build process:
1. Loaded the first 500 HotpotQA questions from the `train` split.
2. Extracted all unique Wikipedia page titles mentioned in `supporting_facts` across those questions.
3. For each title, fetched the page from the Wikipedia API using the `parse` action with `prop=text|links`.
4. Parsed the returned HTML with BeautifulSoup to extract plain text and outgoing hyperlinks.
5. Saved the result incrementally to `data/corpus.json` (every 50 pages) to protect against network interruptions.

The result is approximately 450 pages covering the Wikipedia topics needed to answer 500 HotpotQA multi-hop questions. Below we load this pre-built corpus and inspect its structure.

In [ ]:
# Use pathlib for all file paths — no hardcoded strings (CLAUDE.md requirement)
CORPUS_PATH = PROJECT_ROOT / "data" / "corpus.json"

# Load pre-built corpus (built by src/corpus.py in Phase 2)
# If data/corpus.json doesn't exist, run: python -c "from src.corpus import build_corpus; build_corpus()"
corpus = load_corpus(CORPUS_PATH)
print(f"Corpus: {len(corpus)} Wikipedia pages\n")

# Inspect the structure of one page to understand what each field contains
page = corpus[0]
print(f"Page title:      {page['title']}")
print(f"Text length:     {len(page['text'])} characters (plain text, used by B1)")
print(f"HTML length:     {len(page['html'])} characters (raw HTML, used by B2)")
print(f"Outgoing links:  {len(page['links'])} links (used to build Stage 2 graph)")
print(f"First 5 links:   {page['links'][:5]}")
print(f"\nText preview (first 300 chars):\n{page['text'][:300]}...")

### Stage 1 Summary

We now have two things: a set of HotpotQA questions with gold answers and supporting page references, and a Wikipedia corpus of ~450 pages each containing plain text, raw HTML, and outgoing hyperlinks. The plain text and HTML will be used by the retrieval systems in Stage 4. The outgoing hyperlinks are the raw material for Stage 2: building the graph that powers HyperRAG's 1-hop expansion.

## Stage 2: Graph Construction

**What is a directed graph (DiGraph)?** A directed graph is a set of *nodes* (vertices) connected by *directed edges* (arrows). An edge from node A to node B means "A points to B", but does NOT imply an edge from B back to A. In our hyperlink graph:
- **Nodes** are Wikipedia page titles (e.g. `"Albert Einstein"`).
- **Edges** are hyperlinks: if the Wikipedia page for A contains a `<a href>` link to B, we add a directed edge A → B.

**Why directed?** Hyperlinks are asymmetric. The Wikipedia page for "Theory of Relativity" might link to "Albert Einstein", but the page for "Albert Einstein" might also link back — or might not. The link direction matters because HyperRAG uses *outgoing* links (successors) to expand from a retrieved page to pages it references, not the other way around. Using an undirected graph would conflate these two different relationships.

**Why does this graph matter for HyperRAG?** Multi-hop questions require finding connections between concepts that no single page explicitly states. When we retrieve a page via semantic search, that page's outgoing hyperlinks often point to the *next* page the question requires. HyperRAG follows those links one hop outward — exactly as a human researcher would click to the linked page to continue researching. We call this the "1-hop graph expansion": starting from k semantically similar pages, expand to all their successors in the DiGraph, then re-rank the combined set.

**How we build it:** `src/graph.py` uses a two-pass algorithm. Pass 1: add all corpus page titles as nodes. Pass 2: for each page, iterate through its `links` list; if a link target is also a node in the graph (i.e., also in our corpus), add a directed edge. We only add edges between pages we have fetched — we cannot follow links to pages outside our corpus.

In [ ]:
# Build directed graph from corpus hyperlinks
# Only edges between pages WITHIN our corpus are created
# (We can't follow links to pages we haven't fetched)
graph = build_graph(corpus)

# print_graph_stats shows node count, edge count, density, and top-degree nodes
print_graph_stats(graph)

print(f"\nGraph type:   {type(graph).__name__}  (DiGraph = directed)")
print(f"Is directed:  {graph.is_directed()}")

### Step 2.1: Exploring Graph Structure

We can inspect individual nodes to understand their connectivity. The most important operation for HyperRAG is `G.successors(node)`, which returns all pages that a given page links TO. These are the "1-hop expansion targets": when HyperRAG retrieves a page, it calls `G.successors(page_title)` to discover which pages should be added to the candidate set. The complementary operation is `G.predecessors(node)`, which returns pages that link TO this page — useful for understanding which pages are heavily cited across the corpus.

In [ ]:
# Find a page that has at least 2 outgoing links within our corpus
# (some pages link only to pages not in our subset, so we search)
example_node = None
for node in graph.nodes():
    successors = list(graph.successors(node))
    if len(successors) >= 2:  # need at least 2 for an interesting visualization
        example_node = node
        break

if example_node:
    successors = list(graph.successors(example_node))
    predecessors = list(graph.predecessors(example_node))
    print(f"Page: '{example_node}'")
    print(f"  Links TO (successors):   {successors[:10]}")
    print(f"  Links FROM (predecessors): {predecessors[:10]}")
    print(f"\n  Out-degree: {graph.out_degree(example_node)}  (outgoing hyperlinks to other corpus pages)")
    print(f"  In-degree:  {graph.in_degree(example_node)}  (other corpus pages that link here)")
    # This asymmetry is WHY we use a DiGraph — hyperlinks are not symmetric
    print(f"\n  Note: out-degree != in-degree in general — hyperlinks are NOT symmetric")
else:
    print("No node with 2+ successors found — corpus may be very small")

### Step 2.2: Visualizing the Graph Structure

The visualization below shows a *subgraph* centered on an example Wikipedia page and its immediate neighbors in the hyperlink graph. Each node is a Wikipedia page title; each arrow is a directed hyperlink. We use three colors:

- **Red** — the center page (the one our semantic search retrieved for a query).
- **Orange** — 1-hop successors: pages the center page links TO. These are the pages HyperRAG adds to the candidate set during graph expansion. This is the core HyperRAG operation.
- **Blue** — predecessors: pages that link TO the center page. These are NOT part of the HyperRAG expansion (we follow outgoing links, not incoming links), but they show how the page fits into the broader Wikipedia citation network.

The key insight to take from this visualization: by following just one hop of outgoing links from the red node, we immediately discover the orange nodes — pages that may contain the "second half" of a multi-hop answer that a direct semantic search would never retrieve.

In [ ]:
# --- Graph Structure Visualization ---
# Show a subgraph centered on example_node and its immediate neighbors

if example_node:
    # Collect the subgraph: center + up to 6 successors + up to 6 predecessors
    succ_set = set(list(graph.successors(example_node))[:6])
    pred_set = set(list(graph.predecessors(example_node))[:6])
    neighbors = succ_set | pred_set | {example_node}
    subgraph = graph.subgraph(neighbors)

    fig, ax = plt.subplots(figsize=(12, 8))

    # Color code: center=red, successors=orange (HyperRAG expansion targets), predecessors=blue
    node_colors = []
    for n in subgraph.nodes():
        if n == example_node:
            node_colors.append("#e74c3c")   # red — retrieved/center page
        elif n in succ_set:
            node_colors.append("#f39c12")   # orange — 1-hop expansion targets (HyperRAG adds these)
        else:
            node_colors.append("#3498db")   # blue — predecessors (link to center, not expanded)

    # Deterministic spring layout so the diagram is reproducible
    pos = nx.spring_layout(subgraph, seed=42, k=2)

    # Draw with directed arrows to make the hyperlink direction visible
    nx.draw(
        subgraph, pos, ax=ax,
        node_color=node_colors,
        node_size=1500,
        font_size=7,
        font_weight="bold",
        with_labels=True,
        arrows=True,
        edge_color="#cccccc",
        arrowsize=15,
        connectionstyle="arc3,rad=0.1",  # slight curve so bidirectional edges are visible
    )

    # Legend explaining the color scheme
    legend_elements = [
        mpatches.Patch(color="#e74c3c", label="Center page (retrieved by semantic search)"),
        mpatches.Patch(color="#f39c12", label="1-hop successors (HyperRAG expansion targets)"),
        mpatches.Patch(color="#3498db", label="Predecessors (link TO center — not expanded)"),
    ]
    ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
    ax.set_title(f"Hyperlink Graph Subgraph Around '{example_node}'", fontsize=13)

    plt.tight_layout()
    plt.show()  # inline display only — no file saving (D-06)
else:
    print("No suitable node found for visualization — corpus may be too small")

### Step 2.3: Full Pipeline Overview

Before we move to Stage 3 (Embeddings & FAISS), here is the complete HyperRAG pipeline at a glance. The four stages are sequential: each stage produces output that the next stage consumes. The pipeline is designed so that expensive stages (corpus fetching in Stage 1, index building in Stage 3) are cached to disk and only run once.

In [ ]:
# --- Pipeline Flow Diagram ---
# Matplotlib diagram showing the 4 stages as labeled boxes with directional arrows
# No external diagram libraries needed — pure matplotlib patches

fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis("off")  # hide axes — we're drawing manually

# Each tuple: (x_center, box_width, main_label, subtitle_label)
stages = [
    (1.5,  2.5, "Stage 1\nCorpus Loading",      "HotpotQA + Wikipedia"),
    (5.0,  2.5, "Stage 2\nGraph Construction",   "NetworkX DiGraph"),
    (8.5,  2.5, "Stage 3\nEmbeddings + FAISS",   "Vector similarity search"),
    (12.0, 2.5, "Stage 4\nRetrieval + Eval",      "B1 / B2 / HyperRAG"),
]

# One distinct color per stage so graders can map prose to diagram at a glance
colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"]

for (x, w, label, subtitle), color in zip(stages, colors):
    # Draw rounded rectangle for each stage
    rect = mpatches.FancyBboxPatch(
        (x - w / 2, 0.5), w, 1.8,
        boxstyle="round,pad=0.15",
        facecolor=color, edgecolor="white",
        alpha=0.85, linewidth=2,
    )
    ax.add_patch(rect)
    # Main stage label (bold)
    ax.text(x, 1.6, label, ha="center", va="center",
            fontsize=10, fontweight="bold", color="white")
    # Subtitle / technology note (italic)
    ax.text(x, 0.85, subtitle, ha="center", va="center",
            fontsize=8, color="white", style="italic")

# Arrows connecting stages left-to-right
for i in range(3):
    x_start = stages[i][0] + stages[i][1] / 2 + 0.1    # right edge of box i
    x_end   = stages[i+1][0] - stages[i+1][1] / 2 - 0.1  # left edge of box i+1
    ax.annotate(
        "", xy=(x_end, 1.4), xytext=(x_start, 1.4),
        arrowprops=dict(arrowstyle="->", color="#333333", lw=2),
    )

ax.set_title("HyperRAG Pipeline: From Raw Data to Evaluation", fontsize=13, pad=15)
plt.tight_layout()
plt.show()  # inline display only — no file saving (D-06)

### Stage 2 Summary

We now have a directed hyperlink graph connecting Wikipedia pages in our corpus. The key structural insight is that this graph's edges encode the same navigational paths a human researcher would follow: starting from a retrieved page, follow its outgoing links (successors in the DiGraph) to discover adjacent pages that contain complementary information. This is exactly what HyperRAG does in Stage 4 — retrieve k pages via dense vector search, then expand to their successors, then re-rank the combined set.

In Stage 3, we will encode every corpus page as a dense vector so that we can find the k pages most semantically similar to a query in milliseconds — regardless of whether the query contains the same words as the page text.